# 15 — LeHami: symplectic structure and evidence

**Lesson type: mathematical analogue with partial paper-linked evidence.** This notebook does not implement LeHami or reproduce the paper's benchmark.

## Learning outcome

- Explain what a symplectic update preserves.
- Check that property with a small harmonic-oscillator calculation.
- Separate a structural guarantee from prediction error, planning, and physical accuracy.
- Read one completed local result without calling it a three-seed reproduction.

## Before you start

You need basic algebra and Python. The calculation uses only the Python standard library. It needs no GPU, data, checkpoint, download, or robot hardware.

The local pendulum values below are recorded measurements from the [project run report](../docs/reproductions/lehami-actuated-velocity.md). The code in this notebook does not recompute them. The toy oscillator is only a math example.

## 1. What does symplectic mean?

A **phase-space state** stores a position-like value `q` and a momentum-like value `p`. For one degree of freedom, the canonical symplectic matrix is

$$
J = \begin{bmatrix}0 & 1\\-1 & 0\end{bmatrix}.
$$

A differentiable update with Jacobian `M` is symplectic when `Mᵀ J M = J`. In two dimensions this also means `det(M) = 1`, so the update preserves signed area. It can rotate, stretch, and shear a region. It cannot shrink or grow its total area.

This is a statement about the map. It does not say that each predicted point is accurate, or that `q` and `p` describe a real physical state.

In [1]:
import math


def leapfrog_matrix(step_size):
    """Leapfrog map for H(q, p) = (q*q + p*p) / 2."""
    h = float(step_size)
    if h <= 0:
        raise ValueError("step_size must be positive")
    return (
        (1.0 - h * h / 2.0, h),
        (-h * (1.0 - h * h / 4.0), 1.0 - h * h / 2.0),
    )


def explicit_euler_matrix(step_size):
    """Explicit Euler map for q_dot=p and p_dot=-q."""
    h = float(step_size)
    return ((1.0, h), (-h, 1.0))


def determinant(matrix):
    return matrix[0][0] * matrix[1][1] - matrix[0][1] * matrix[1][0]


def transform_points(points, matrix):
    return [
        (
            matrix[0][0] * q + matrix[0][1] * p,
            matrix[1][0] * q + matrix[1][1] * p,
        )
        for q, p in points
    ]


def polygon_area(points):
    """Return polygon area using the shoelace formula."""
    if len(points) < 3:
        raise ValueError("a polygon needs at least three vertices")
    total = 0.0
    for index, (q, p) in enumerate(points):
        next_q, next_p = points[(index + 1) % len(points)]
        total += q * next_p - next_q * p
    return abs(total) / 2.0


def circle_polygon(vertex_count=256):
    return [
        (
            math.cos(2.0 * math.pi * index / vertex_count),
            math.sin(2.0 * math.pi * index / vertex_count),
        )
        for index in range(vertex_count)
    ]


## 2. Check one leapfrog step

For this oscillator, Hamilton's equations are `q_dot = p` and `p_dot = -q`. Leapfrog applies a half kick, a full drift, then a half kick:

1. `p_half = p - h*q/2`
2. `q_next = q + h*p_half`
3. `p_next = p_half - h*q_next/2`

These steps produce the matrix in the previous cell. The next cell checks its determinant and the area of a 256-sided region. Python uses double-precision floats here, so a tiny roundoff error is expected. The accepted absolute error is `1e-12`.

In [2]:
STEP_SIZES = (0.2, 0.8, 2.2)
AREA_TOLERANCE = 1e-12
polygon = circle_polygon()
area_before = polygon_area(polygon)

print("step  det(M)              |det(M)-1|         area ratio          |ratio-1|")
for step_size in STEP_SIZES:
    matrix = leapfrog_matrix(step_size)
    determinant_error = abs(determinant(matrix) - 1.0)
    area_after = polygon_area(transform_points(polygon, matrix))
    area_ratio = area_after / area_before
    area_error = abs(area_ratio - 1.0)
    print(
        f"{step_size:>4.1f}  {determinant(matrix):.16f}  "
        f"{determinant_error:.3e}       {area_ratio:.16f}  {area_error:.3e}"
    )
    assert determinant_error <= AREA_TOLERANCE
    assert area_error <= AREA_TOLERANCE

print(f"\nAll checks pass with absolute tolerance {AREA_TOLERANCE:g}.")


step  det(M)              |det(M)-1|         area ratio          |ratio-1|
 0.2  0.9999999999999999  1.110e-16       1.0000000000000004  4.441e-16
 0.8  1.0000000000000000  0.000e+00       1.0000000000000007  6.661e-16
 2.2  0.9999999999999998  2.220e-16       1.0000000000000024  2.442e-15

All checks pass with absolute tolerance 1e-12.


The determinant and polygon area stay at one for all three step sizes, up to floating-point roundoff. This is the structural property that the paper builds into each LeHami leapfrog step.

Now compare explicit Euler. This is only a contrast between two numerical updates. It is **not** the LeWM model or a simulation of LeWM.

In [3]:
print("step  det(Euler)         area ratio")
for step_size in (0.2, 0.8):
    matrix = explicit_euler_matrix(step_size)
    area_ratio = polygon_area(transform_points(polygon, matrix)) / area_before
    print(f"{step_size:>4.1f}  {determinant(matrix):.8f}       {area_ratio:.8f}")
    assert abs(determinant(matrix) - (1.0 + step_size * step_size)) <= AREA_TOLERANCE


step  det(Euler)         area ratio
 0.2  1.04000000       1.04000000
 0.8  1.64000000       1.64000000


For explicit Euler, the determinant is `1 + h²`, so its mapped area grows. The contrast shows that a choice of numerical update can impose a structural property. It does not establish how a learned model predicts a pendulum.

## 3. Structure does not guarantee a close trajectory

Even a symplectic update can make a poor numerical prediction when the step size is too large. The exact oscillator stays bounded. Leapfrog with `h = 2.2` is outside the stable step-size range for this oscillator, so its state error grows while its map still preserves area.

The next cell compares each 20-step leapfrog result with the exact oscillator state at the same final time. This is a property of the toy integrator, not a measurement of LeHami.

In [4]:
def exact_oscillator_state(q0, p0, time):
    cosine = math.cos(time)
    sine = math.sin(time)
    return (q0 * cosine + p0 * sine, -q0 * sine + p0 * cosine)


q0, p0 = 1.0, 0.3
step_count = 20
print("step  det(M)     final-time state error")
for step_size in STEP_SIZES:
    matrix = leapfrog_matrix(step_size)
    q, p = q0, p0
    for _ in range(step_count):
        q, p = (
            matrix[0][0] * q + matrix[0][1] * p,
            matrix[1][0] * q + matrix[1][1] * p,
        )
    exact_q, exact_p = exact_oscillator_state(q0, p0, step_count * step_size)
    error = math.hypot(q - exact_q, p - exact_p)
    print(f"{step_size:>4.1f}  {determinant(matrix):.8f}  {error:.6g}")


step  det(M)     final-time state error
 0.2  1.00000000  0.00334234
 0.8  1.00000000  0.416915
 2.2  1.00000000  9.64245e+06


At `h = 2.2`, the determinant remains one but the final-time state error is large. Volume preservation rules out uniform collapse; it does not bound trajectory error. The learned representation can also be poorly aligned with the real system, even when its update has the right geometry.

## 4. How LeHami uses this structure

LeHami predicts over a latent pair `(q, p)`. The encoder gives the position-like component `q`; a fixed causal Savitzky–Golay derivative stencil gives the momentum-like component `p`. A learned separable Hamiltonian uses

$$
H_φ(q,p,u) = K_φ(p) + V_φ(q,u)
$$

and a leapfrog step advances the pair. The action `u` is held fixed within each step. With a changing action sequence, the learned Hamiltonian can change and energy need not stay constant. The symplectic property still applies to each step under the paper's assumptions.

The toy oscillator above uses known coordinates and a known Hamiltonian. LeHami learns its encoder and Hamiltonian from rendered pendulum sequences. The toy calculation does not validate those learned parts.

## 5. Paper result and local measurement

The paper's Table 1 reports held-out angular-velocity MAE at the actuated-pendulum anchor (`K=0`) using a 25-neighbor probe. The paper values summarize three training seeds. The local rows are from one completed seed-0 run per model using the official K=0 probe and data split. They are not local means.

| Source | Model | Training seeds | K=0 25-NN angular-velocity MAE |
| --- | --- | ---: | ---: |
| Paper Table 1 | LeHami | 3 | 0.419 ± 0.045 rad/s |
| Paper Table 1 | LeWM | 3 | 2.508 ± 0.002 rad/s |
| Local full run | LeHami `a0q` | 1 (seed 0) | 0.456258 rad/s |
| Local full run | LeWM `b0s` | 1 (seed 0) | 2.504686 rad/s |

The paper values are reported results. The local values are project measurements. The detailed run record lists the code revisions, model configs, data hash, environment, and compute. See the [paper](https://arxiv.org/abs/2610.15267), the [official code](https://github.com/rominay/LeHami), and the [local run report](../docs/reproductions/lehami-actuated-velocity.md).

### K=1-fitted rollout probe (one local seed)

The paper's Figure 15 reports velocity error over the actuated-pendulum rollout with a 25-neighbor probe fitted at K=1 and held fixed. The local run uses that probe setup with each saved seed-0 checkpoint and reports the held-out test split. These values are separate from the K=0 Table 1 values above.

| Rollout step K | LeHami `a0q` MAE (rad/s) | LeWM `b0s` MAE (rad/s) |
| ---: | ---: | ---: |
| 1 | 0.467665 | 0.513117 |
| 5 | 0.606430 | 0.811439 |
| 10 | 1.033282 | 0.869993 |
| 20 | 1.393730 | 1.232498 |
| 30 | 1.640654 | 1.481048 |
| 50 | 2.188266 | 1.666773 |
| 76 | 2.881099 | 2.552452 |

For this seed, LeHami has lower error at K=1 and K=5; LeWM has lower error at the selected steps from K=10 onward. This is a one-seed observation, not a reproduction of the paper's three-seed Figure 15 result. The full per-step values, run conditions, and limits are in the [run report](../docs/reproductions/lehami-actuated-velocity.md).

## 6. What this evidence supports

- **Structural claim:** The ideal leapfrog map used in the example preserves phase-space area. The LeHami paper proves the corresponding symplectic property for its action-conditioned step under its stated assumptions.
- **Measured claim:** The local seed-0 K=0 probes measured the values in the table. A separate K=1-fitted, frozen 25-neighbor probe measured held-out velocity error through K=76 for both saved checkpoints. Both results use the specified actuated-pendulum split.
- **Still open:** Four full training runs remain: seeds 1 and 2 for each model. Until they finish, this project has no local three-seed mean or standard deviation and does not call the paper finding reproduced.
- **Not tested by these probes:** Action selection, planning or control performance, and physical-system validity. Prediction error alone does not establish these outcomes.

The paper's structural guarantee is useful, but it is one part of the evidence. A map can preserve volume and still have inaccurate trajectories.

## Sources and licenses

- Yalovetzky, Balestriero, and LeCun, [LeHami: Symplectic Latent Dynamics for World Models](https://arxiv.org/abs/2610.15267), v1. The paper is distributed under CC BY 4.0.
- Authors' [official LeHami repository](https://github.com/rominay/LeHami), licensed Apache-2.0.
- [Local actuated-pendulum run report](../docs/reproductions/lehami-actuated-velocity.md).

This notebook links to the paper and source code. It does not copy their figures, code, or data.

## Check your understanding

1. Why can the area stay constant while the trajectory becomes inaccurate?
2. What assumption about the action is made inside each LeHami step?
3. Which additional measurements would be needed before making a claim about planning or control?

Try changing `STEP_SIZES`. The determinant and area checks should still pass. Compare that structural result with the final-time error.